# Exercice Juin 2025 - DHIS2 extraction & DHIS2 extraction pipeline
Premier essai dans VSC
Objectif = 
- Code qui va chercher des certains DataElements  (liste de 10 DataElements) , les extraire, pendant 4 périodes différentes.C
- Ces 10 éléments doivent etre dans des DataSets différents,
- Les périodes ne doivent pas être continue (eg. Janvier, Avril, Juin, Aout)
- Pour certaines OU qui sont dans un Orgunit Groups (ou list manuel).
- Le faire pour un niveau (FOSA) puis en agrégant puis District. Parfois à des niveau différents FOSA et District
- Petite analyse en conclusion
- Faire des push

Partie 1 - Connection à DHIS2

In [41]:
USERNAME = "admin"
PASSWORD = "district"
base_url = "https://play.im.dhis2.org/stable-2-39-9-1/api/29/"

In [42]:
# Extraction de la liste de tous les dataElements

import pandas as pd
import requests
url = f"{base_url}dataElements"

params = {
    "paging": "false",
    "fields": "id,name,domainType"
}
auth = (USERNAME,PASSWORD)  # Corrected variable name from PASSWORD to PASSWORD

response = requests.get(url, params=params, auth=auth)
data = response.json()
DEs = pd.DataFrame(data["dataElements"])
print(f"Nombre de dataElements : {len(DEs)}")
DEs.head(10)

Nombre de dataElements : 1036


,name,domainType,id
0,Accute Flaccid Paralysis (Deaths < 5 yrs),AGGREGATE,FTRrcoaog83
1,Acute Flaccid Paralysis (AFP) follow-up,AGGREGATE,P3jJH5Tu5VC
2,Acute Flaccid Paralysis (AFP) new,AGGREGATE,FQ2o8UBlcrS
3,Acute Flaccid Paralysis (AFP) referrals,AGGREGATE,M62VHgYT2n0
4,Additional medication,TRACKER,WO8yRIZb7nb
5,Additional notes related to facility,AGGREGATE,uF1DLnZNlWe
6,Admission Date,TRACKER,eMyVanycQSC
7,Age in years,TRACKER,qrur9Dvnyt5
8,Age of LLINs,TRACKER,JuTpJ2Ywq5b
9,Albendazole given at ANC (2nd trimester),AGGREGATE,hCVSHjcml9g


In [43]:
# Choix de 10 DATALEMENTS pour l'exercice

DE_selected = ["ANC 1st visit", "ANC 2nd visit", "ANC 3rd visit", "ANC 4th or more visits",
                 "Total Population", "Total population < 1 year", "Total population < 5 years", "Population of women of child bearing age (WRA)",
                 "Inpatient malaria cases", "Prevalence of tuberculosis (per 100 000 population)"]


for DE in DE_selected:
    if DE not in DEs["name"].values:
        raise ValueError(f"Data element '{DE}' not found in the list of data elements.")

# Filtrage des dataElements pour ne garder que ceux qui sont dans DE_selecte

DE_selected = DEs[DEs["name"].isin(DE_selected)]

# Vérification que les dataElements sélectionnés sont de type AGGREGATE
for _, DE in DE_selected.iterrows():
    if DE["domainType"] ==  "TRACKER":
        raise ValueError(
            f"Data element '{DE['name']}' is a TRACKER DE.\n Please choose only AGGREGATE data elements.")
print(f"Nombre de dataElements sélectionnés : {len(DE_selected)}")
DE_selected.head(10)

Nombre de dataElements sélectionnés : 10


,name,domainType,id
19,ANC 1st visit,AGGREGATE,fbfJHSPpUQD
20,ANC 2nd visit,AGGREGATE,cYeuwXTCPkU
21,ANC 3rd visit,AGGREGATE,Jtf34kNZhzP
22,ANC 4th or more visits,AGGREGATE,hfdmMSPBgLG
325,Inpatient malaria cases,AGGREGATE,p4K11MFEWtw
639,Population of women of child bearing age (WRA),AGGREGATE,vg6pdjObxsm
652,Prevalence of tuberculosis (per 100 000 popula...,AGGREGATE,sTzxtq72Moq
839,Total Population,AGGREGATE,WUg3MYWQ7pt
840,Total population < 1 year,AGGREGATE,DTVRnCGamkV
841,Total population < 5 years,AGGREGATE,DTtCy7Nx5jH


# Extraction des valeurs des DEs sélectionné

In [44]:
url = f"{base_url}dataValueSets"

params = {
    "dataElement": ",".join(DE_selected["id"].tolist()),
    "period": ["202501", "202504", "202506"],      
    "orgUnitGroup": "oRVt7g429ZO",
}

response = requests.get(url, params=params, auth=auth)
data = response.json()

print("hello")
print ("data")
DEs_values = pd.DataFrame(data["dataValues"])
print(f"Nombre de valeurs de dataElements : {len(DEs_values)}")

hello
data
Nombre de valeurs de dataElements : 16867


In [45]:
DEs_values.head(10)

,dataElement,period,orgUnit,categoryOptionCombo,attributeOptionCombo,value,storedBy,created,lastUpdated,comment,followup
0,fbfJHSPpUQD,202501,PuZOFApTSeo,pq2XI5kz2BY,HllvX50cXC0,13,bo2,2010-03-05T00:00:00.000+0000,2010-03-05T00:00:00.000+0000,,False
1,fbfJHSPpUQD,202506,sLKHXoBIqSs,pq2XI5kz2BY,HllvX50cXC0,4,kono1,2010-07-26T00:00:00.000+0000,2010-07-26T00:00:00.000+0000,,False
2,fbfJHSPpUQD,202501,erqWTArTsyJ,pq2XI5kz2BY,HllvX50cXC0,15,bo2,2010-03-09T00:00:00.000+0000,2010-03-09T00:00:00.000+0000,,False
3,fbfJHSPpUQD,202501,egv5Es0QlQP,PT59n8BQbqM,HllvX50cXC0,1,bo2,2010-03-13T00:00:00.000+0000,2010-03-13T00:00:00.000+0000,,False
4,fbfJHSPpUQD,202501,KvE0PYQzXMM,PT59n8BQbqM,HllvX50cXC0,4,bo2,2010-03-17T00:00:00.000+0000,2010-03-17T00:00:00.000+0000,,False
5,fbfJHSPpUQD,202501,ZpE2POxvl9P,PT59n8BQbqM,HllvX50cXC0,8,bo2,2010-03-17T00:00:00.000+0000,2010-03-17T00:00:00.000+0000,,False
6,fbfJHSPpUQD,202501,ZpE2POxvl9P,pq2XI5kz2BY,HllvX50cXC0,7,bo2,2010-03-17T00:00:00.000+0000,2010-03-17T00:00:00.000+0000,,False
7,fbfJHSPpUQD,202501,mGmu0GJ5neg,pq2XI5kz2BY,HllvX50cXC0,21,bo2,2010-03-13T00:00:00.000+0000,2010-03-13T00:00:00.000+0000,,False
8,fbfJHSPpUQD,202501,RhJbg8UD75Q,pq2XI5kz2BY,HllvX50cXC0,8,bo2,2010-03-10T00:00:00.000+0000,2010-03-10T00:00:00.000+0000,,False
9,fbfJHSPpUQD,202501,EFTcruJcNmZ,pq2XI5kz2BY,HllvX50cXC0,10,bo2,2010-03-04T00:00:00.000+0000,2010-03-04T00:00:00.000+0000,,False
